# DeskPilot v0.1 — Your First Tool-Calling Agent
**Techademy — Build Your First AI Agent | Module 2 Live Build**

Meridian Corp's Tier-1 IT help desk is drowning in repetitive tickets. In this notebook you'll build **DeskPilot**, an agent that can reason about an incoming ticket and *take action* by calling real functions — not just suggest what a human should do.

This notebook uses **Groq** as the LLM provider — an OpenAI-compatible chat API that runs open models (Llama 3.3) at very high inference speed, which makes the tool-calling loop feel snappy even with several round trips.

By the end you will have:
1. A connection to Groq
2. A system prompt that defines DeskPilot's persona and scope
3. Two tools DeskPilot can call: `create_ticket` and `check_vpn_status`
4. A Reason + Act (ReAct) loop that lets the model choose and call tools
5. Three simulated tickets run end to end
6. An exercise: add a third tool on your own

## Step 0 — Before you start
1. **Runtime → Change runtime type** → keep the default CPU runtime (no GPU needed).
2. Get a free API key at [console.groq.com/keys](https://console.groq.com/keys).
3. Open the **Secrets** panel (key icon in the left sidebar) and add a secret named `GROQ_API_KEY` with your key. Toggle **notebook access** on.
4. Run the cells top to bottom.

## Step 1 — Install dependencies

In [1]:
!pip install -q groq

In [2]:
import openai

## Step 2 — Connect to Groq

In [3]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"  # supports tool calling

# Quick connectivity check
ping = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
)
print(ping.choices[0].message.content)

Connected


In [4]:
from google.colab import userdata
from groq import Groq
from openai import OpenAI # Import OpenAI

# Get API keys
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") # OpenAI key is optional for fallback

# Initialize clients
groq_client = Groq(api_key=GROQ_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# Define models - primary is Groq, fallback is OpenAI
PRIMARY_MODEL = "llama3-8b-8192" # A common Groq model
FALLBACK_MODEL = "gpt-3.5-turbo" # A common OpenAI model

print(f"Primary LLM: Groq with model '{PRIMARY_MODEL}'")
if openai_client:
    print(f"Fallback LLM: OpenAI with model '{FALLBACK_MODEL}'")
else:
    print("OpenAI client not initialized (no OPENAI_API_KEY found). No fallback.")

# Quick connectivity check for primary client
try:
    ping = groq_client.chat.completions.create(
        model=PRIMARY_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
    )
    print(f"Groq primary client status: {ping.choices[0].message.content}")
except Exception as e:
    print(f"Groq primary client connection failed: {e}")

# If openai_client exists, perform a quick connectivity check for it too.
if openai_client:
    try:
        ping_openai = openai_client.chat.completions.create(
            model=FALLBACK_MODEL,
            messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
        )
        print(f"OpenAI fallback client status: {ping_openai.choices[0].message.content}")
    except Exception as e:
        print(f"OpenAI fallback client connection failed: {e}")

Primary LLM: Groq with model 'llama3-8b-8192'
Fallback LLM: OpenAI with model 'gpt-3.5-turbo'
Groq primary client connection failed: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
OpenAI fallback client status: Connected


In [5]:
import json
import time

def call_llm(client_obj, model_name, messages, tools):
    """Generic function to call an LLM client."""
    return client_obj.chat.completions.create(
        model=model_name, messages=messages, tools=tools
    )

def call_model_with_retry(client_obj, model_name, messages, tools, max_retries=3):
    """Call the chat API with simple exponential backoff on transient errors."""
    delay = 1.5
    for attempt in range(max_retries):
        try:
            return call_llm(client_obj, model_name, messages, tools)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"  [transient error: {e} — retrying in {delay:.1f}s] using {model_name}")
            time.sleep(delay)
            delay *= 2

def run_deskpilot(user_message, employee_id, verbose=True, max_steps=6):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[employee_id={employee_id}] {user_message}"},
    ]

    current_client = groq_client
    current_model = PRIMARY_MODEL

    for step in range(max_steps):
        response = None
        try:
            response = call_model_with_retry(current_client, current_model, messages, TOOLS)
        except Exception as e_primary:
            print(f"Primary LLM ({current_model}) call failed: {e_primary}")
            if openai_client:
                print(f"Attempting fallback to OpenAI ({FALLBACK_MODEL})...")
                current_client = openai_client
                current_model = FALLBACK_MODEL
                try:
                    response = call_model_with_retry(current_client, current_model, messages, TOOLS)
                except Exception as e_fallback:
                    print(f"Fallback LLM ({current_model}) call also failed: {e_fallback}")
                    return "[DeskPilot failed to get a response from any LLM — escalate to a human]"
            else:
                return "[DeskPilot failed to get a response from primary LLM and no fallback is configured — escalate to a human]"

        if response is None: # This should not happen if errors are caught or retries succeed
             return "[DeskPilot encountered an unexpected error during LLM call — escalate to a human]"


        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[Final Answer] {msg.content}\n")
            return msg.content

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            if verbose:
                print(f"[Action] {name}({tool_call.function.arguments})")

            try:
                args = json.loads(tool_call.function.arguments)
                if name not in TOOL_IMPL:
                    raise ValueError(f"Unknown tool: {name}")
                result = TOOL_IMPL[name](**args)
            except Exception as e:
                result = {"error": str(e)}

            if verbose:
                print(f"[Observation] {result}\n")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return "[DeskPilot hit the step limit — escalate to a human]"

## Step 3 — DeskPilot's system prompt
This is where the persona, scope, and boundaries live — before DeskPilot ever sees a ticket.

In [6]:
SYSTEM_PROMPT = """
You are DeskPilot, Meridian Corp's Tier-1 IT support agent.

- Resolve password resets, VPN access, and software install requests.
- Ask one clarifying question if the request is ambiguous.
- Use the available tools rather than guessing at ticket or account status.
- If a request falls outside IT support, or needs manager approval,
  say so plainly and recommend escalating to a human agent instead of attempting it.
- Keep responses under 4 sentences, plain and professional.
"""

## Step 4 — Mock enterprise systems
In a real deployment these functions would call Meridian's ITSM and identity systems. Here we simulate them with a small in-memory "database" so the notebook runs standalone.

In [7]:
import random
import uuid

# --- fake backend state -----------------------------------------------------
# For READ
_VPN_STATUS = {
    "E1001": {"status": "locked", "failed_attempts": 3},
    "E1002": {"status": "active", "failed_attempts": 0},
    "E1003": {"status": "locked", "failed_attempts": 5},
}

# For CREATE
_TICKETS = []

# For Update
_PASSWORDS = {"E1001": "abc123",
              "E1002": "xyz123"}

# For Logs - updated to include service_name
_LOGS = {
    "REQ-12345": {
        "service_name": "auth-service",
        "log": "User login successful. Timestamp: 2024-05-15 10:00:00. IP: 192.168.1.10"
    },
    "REQ-67890": {
        "service_name": "payment-gateway",
        "log": "Database query failed. Error: Connection refused. Timestamp: 2024-05-15 10:05:30."
    },
    "REQ-11111": {
        "service_name": "auth-service",
        "log": "Password reset initiated for user E1001. Timestamp: 2024-05-15 11:00:00."
    },
    "REQ-22222": {
        "service_name": "payment-gateway",
        "log": "Transaction processed successfully. Amount: $50.00. Timestamp: 2024-05-15 11:05:00."
    }
}

def create_ticket(category: str, summary: str, employee_id: str) -> dict:
    """Open a new Tier-1 IT ticket in the (simulated) service desk system."""
    ticket_id = f"TCK-{uuid.uuid4().hex[:6].upper()}"
    ticket = {
        "ticket_id": ticket_id,
        "category": category,
        "summary": summary,
        "employee_id": employee_id,
        "status": "open",
    }
    _TICKETS.append(ticket)
    return {"ticket_id": ticket_id, "status": "open"}


def check_vpn_status(employee_id: str) -> dict:
    """Check whether an employee's VPN account is active or locked."""
    record = _VPN_STATUS.get(employee_id)
    if not record:
        return {"error": f"No VPN record found for {employee_id}"}
    return {"employee_id": employee_id, **record}

def reset_password(employee_id: str) -> dict:
    """Reset an employee's password."""
    if employee_id not in _PASSWORDS:
        return {"error": f"No password record found for {employee_id}"}

    NEW_PASSWORD = str(random.randint(100000, 999999))
    _PASSWORDS[employee_id] = NEW_PASSWORD
    return {"employee_id": employee_id, "new_password": NEW_PASSWORD}

def get_logs(request_id: str = None, service_name: str = None, keyword: str = None) -> dict:
    """Retrieve logs for a specific production request, with optional service filtering and keyword search."""
    filtered_logs = []

    if request_id:
        log_entry = _LOGS.get(request_id)
        if log_entry:
            logs_to_process = [{
                "request_id": request_id,
                "service_name": log_entry["service_name"],
                "log": log_entry["log"]
            }]
        else:
            return {"error": f"No logs found for request ID {request_id}"}
    else:
        # If no specific request_id, process all logs for filtering
        logs_to_process = [
            {"request_id": k, "service_name": v["service_name"], "log": v["log"]}
            for k, v in _LOGS.items()
        ]

    for log_data in logs_to_process:
        match_service = True
        if service_name and log_data["service_name"].lower() != service_name.lower():
            match_service = False

        match_keyword = True
        if keyword and keyword.lower() not in log_data["log"].lower():
            match_keyword = False

        if match_service and match_keyword:
            filtered_logs.append(log_data)

    if not filtered_logs:
        return {"message": "No matching logs found.", "request_id": request_id, "service_name": service_name, "keyword": keyword}
    return {"logs": filtered_logs}

print("Mock backend ready:", len(_VPN_STATUS), "employee VPN records")

Mock backend ready: 3 employee VPN records


## Step 5 — Describe the tools to the model
Each tool is described with a JSON schema. The model decides *when* to call it and *what* arguments to pass — we never hard-code the decision.

In [8]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "create_ticket",
            "description": "Open a new Tier-1 IT ticket in the service desk system",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "enum": ["password", "vpn", "software"]},
                    "summary": {"type": "string"},
                    "employee_id": {"type": "string"},
                },
                "required": ["category", "summary", "employee_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_vpn_status",
            "description": "Check whether an employee's VPN account is active or locked",
            "parameters": {
                "type": "object",
                "properties": {"employee_id": {"type": "string"}},
                "required": ["employee_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "reset_password",
            "description": "Reset an employee's password",
            "parameters": {
                "type": "object",
                "properties": {"employee_id": {"type": "string"}},
                "required": ["employee_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_logs",
            "description": "Retrieve logs for a specific production request, with optional service filtering and keyword search.",
            "parameters": {
                "type": "object",
                "properties": {
                    "request_id": {"type": "string", "description": "Optional: The ID of the specific production request."
                    },
                    "service_name": {"type": "string", "description": "Optional: Filter logs by the name of the service (e.g., 'auth-service', 'payment-gateway')."
                    },
                    "keyword": {"type": "string", "description": "Optional: Search for a specific keyword within the log content."
                    }
                },
                "required": [], # All parameters are optional, but at least one should ideally be provided by the model
            },
        },
    }
]

# Map tool names to the Python functions that implement them
TOOL_IMPL = {
    "create_ticket": create_ticket,
    "check_vpn_status": check_vpn_status,
    "reset_password": reset_password,
    "get_logs": get_logs,
}


## Step 6 — The ReAct loop
The agent alternates: ask the model what to do next -> if it wants a tool, run the tool and feed the result back -> repeat until the model produces a final answer with no more tool calls.

This also includes basic **error handling**: invalid arguments or an unknown tool are reported back to the model as an observation, and transient API errors are retried with backoff.

In [9]:
import json
import time


def call_model_with_retry(messages, tools, max_retries=3):
    """Call the chat API with simple exponential backoff on transient errors."""
    delay = 1.5
    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools
            )
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"  [transient error: {e} — retrying in {delay:.1f}s]")
            time.sleep(delay)
            delay *= 2


def run_deskpilot(user_message, employee_id, verbose=True, max_steps=6):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[employee_id={employee_id}] {user_message}"},
    ]

    for step in range(max_steps):
        response = call_model_with_retry(messages, TOOLS)
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[Final Answer] {msg.content}\n")
            return msg.content

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            if verbose:
                print(f"[Action] {name}({tool_call.function.arguments})")

            try:
                args = json.loads(tool_call.function.arguments)
                if name not in TOOL_IMPL:
                    raise ValueError(f"Unknown tool: {name}")
                result = TOOL_IMPL[name](**args)
            except Exception as e:
                result = {"error": str(e)}

            if verbose:
                print(f"[Observation] {result}\n")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return "[DeskPilot hit the step limit — escalate to a human]"

## Step 7 — Run three sample tickets

In [10]:
run_deskpilot("My VPN keeps disconnecting, can you check what's going on?", employee_id="E1001")

[Action] check_vpn_status({"employee_id":"E1001"})
[Observation] {'employee_id': 'E1001', 'status': 'locked', 'failed_attempts': 3}

[Final Answer] Your VPN account is currently locked due to multiple failed attempts. I can submit a ticket to unlock it—shall I proceed?



'Your VPN account is currently locked due to multiple failed attempts. I can submit a ticket to unlock it—shall I proceed?'

In [11]:
run_deskpilot("I need Slack installed on my laptop.", employee_id="E1002")

[Action] create_ticket({"category":"software","employee_id":"E1002","summary":"Install Slack on laptop"})
[Observation] {'ticket_id': 'TCK-DF7BB8', 'status': 'open'}

[Final Answer] Your software installation request has been logged (Ticket TCK-DF7BB8). We'll proceed with installing Slack on your laptop. If you need any further assistance, please let us know.



"Your software installation request has been logged (Ticket\u202fTCK-DF7BB8). We'll proceed with installing Slack on your laptop. If you need any further assistance, please let us know."

In [12]:
run_deskpilot("Can you approve a budget increase for my team's software licenses?", employee_id="E1003")

[Final Answer] I’m unable to approve budget requests. Please forward this to your manager or the appropriate procurement team for further review.



'I’m unable to approve budget requests. Please forward this to your manager or the appropriate procurement team for further review.'

In [13]:
run_deskpilot("Can you reset my password?, print both the early password and new password", employee_id="E1001")

[Action] reset_password({"employee_id":"E1001"})
[Observation] {'employee_id': 'E1001', 'new_password': '155255'}

[Final Answer] Your password has been reset. Your new password is **155255**; please log in and change it at your earliest convenience. For security reasons, we cannot disclose your previous password. If you encounter any issues, let me know.



'Your password has been reset. Your new password is **155255**; please log in and change it at your earliest convenience. For security reasons, we cannot disclose your previous password. If you encounter any issues, let me know.'

Notice the third ticket: DeskPilot recognizes a **budget approval** request is outside its scope (per the system prompt) and declines to attempt it rather than guessing — this is the escalation boundary in action, which we'll formalize with guardrails in Module 4.

## Exercise — add a third tool
Add a `reset_password(employee_id)` tool:
1. Write a mock implementation (return a fake temporary password).
2. Add its JSON schema to `TOOLS`.
3. Register it in `TOOL_IMPL`.
4. Re-run a ticket like *"I forgot my password, can you reset it?"* and confirm DeskPilot calls your new tool.

In [15]:
# Your code here

# Test the new tool
run_deskpilot("Can you get the logs for production request REQ-12345?", employee_id="E1001")
run_deskpilot("I need to see the logs for request REQ-67890.", employee_id="E1002")
run_deskpilot("Retrieve logs for non-existent request ABC-123.", employee_id="E1003")
run_deskpilot("Show me logs for 'auth-service' containing the word 'login'.", employee_id="E1001")
run_deskpilot("Find logs from the 'payment-gateway' service that mention 'failed'.", employee_id="E1002")
run_deskpilot("Are there any logs related to password resets in 'auth-service'?", employee_id="E1001")
run_deskpilot("What about logs for 'non-existent-service' with keyword 'error'?", employee_id="E1003")


[Action] get_logs({"request_id":"REQ-12345"})
[Observation] {'logs': [{'request_id': 'REQ-12345', 'service_name': 'auth-service', 'log': 'User login successful. Timestamp: 2024-05-15 10:00:00. IP: 192.168.1.10'}]}

[Final Answer] Here are the logs for production request **REQ-12345**:

- Service: auth‑service  
- Log: *User login successful. Timestamp: 2024‑05‑15 10:00:00. IP: 192.168.1.10*

[Action] get_logs({"request_id":"REQ-67890"})
[Observation] {'logs': [{'request_id': 'REQ-67890', 'service_name': 'payment-gateway', 'log': 'Database query failed. Error: Connection refused. Timestamp: 2024-05-15 10:05:30.'}]}

[Final Answer] Here are the logs for request **REQ-67890**:

- **Service:** payment-gateway  
- **Log:** Database query failed. Error: Connection refused. Timestamp: 2024-05-15 10:05:30.  

[Final Answer] The request to retrieve logs for a specific production request is outside the scope of Tier‑1 IT support. Please forward this to the appropriate operations team or open a r

'No logs were found for service “non-existent-service” with the keyword “error”.'

In [16]:
# Testing the tool without the required positional argument
run_deskpilot("Can you get me some logs that specifically mention about failed login attempts?")
# Output: TypeError: run_deskpilot() missing 1 required positional argument: 'employee_id'